# 🧠 Notebook 4: CNN Classification Model

## 1D CNN for Brain Tumor Classification (Benign vs Malignant)

---

### Pipeline Stage
```
  DE/IDE Features (90,)             CNN Classifier                Output
  from Notebook 03     ───▶   ┌────────────────────┐    ┌─────────────┐
                              │  Conv1D Layers +   │    │  Benign /    │
                              │  Dense + Sigmoid   │──▶│  Malignant   │
                              └────────────────────┘    └─────────────┘
```

### Objective
Train a **1D Convolutional Neural Network (CNN)** on the differential equation features to classify brain MRI slices as **Benign (LGG)** or **Malignant (HGG)**.

### Model Architecture
- Input: DE/IDE feature vector reshaped for 1D convolution
- Multiple Conv1D blocks with BatchNormalization and Dropout
- Global Average Pooling
- Dense classification head with Sigmoid activation

## 1. Import Dependencies

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.utils import plot_model

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f"✅ TensorFlow version: {tf.__version__}")
print(f"   GPUs available: {len(gpus)}")
if gpus:
    for gpu in gpus:
        print(f"   - {gpu.name}")
    # Allow memory growth
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

## 2. Load DE Features

In [ ]:
OUTPUT_DIR = './processed_data'
MODEL_DIR = './models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Load features from Notebook 03
features_path = os.path.join(OUTPUT_DIR, 'de_features.npz')

if os.path.exists(features_path):
    data = np.load(features_path, allow_pickle=True)
    features = data['features_normalized']  # Use normalized features
    labels = data['labels']
    feature_names = data['feature_names']
    print(f"✅ Loaded DE features from Notebook 03")
else:
    # Generate synthetic features for demo
    print("⚠️ DE features not found. Generating synthetic features...")
    n_samples = 1000
    n_features = 90
    features = np.random.randn(n_samples, n_features).astype(np.float32)
    labels = np.random.randint(0, 2, n_samples)
    
    # Add class-specific patterns
    features[labels == 1, :30] += 0.3  # Malignant has higher values in first 30 features
    features[labels == 0, 30:60] += 0.2  # Benign has different pattern
    feature_names = [f'feature_{i}' for i in range(n_features)]

print(f"   Features shape: {features.shape}")
print(f"   Labels shape  : {labels.shape}")
print(f"   Malignant     : {np.sum(labels == 1)} ({100*np.mean(labels==1):.1f}%)")
print(f"   Benign        : {np.sum(labels == 0)} ({100*np.mean(labels==0):.1f}%)")

## 3. Data Splitting

Split the dataset into **training (70%)**, **validation (15%)**, and **test (15%)** sets with stratified sampling to preserve class distribution.

In [ ]:
# First split: train+val (85%) vs test (15%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    features, labels, test_size=0.15, random_state=42, stratify=labels
)

# Second split: train (70% of original) vs val (15% of original)
# 15/85 ≈ 0.176
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.176, random_state=42, stratify=y_trainval
)

print(f"📊 Data Split Summary:")
print(f"   Training  : {X_train.shape[0]:>6d} samples ({100*len(X_train)/len(features):.1f}%)")
print(f"   Validation: {X_val.shape[0]:>6d} samples ({100*len(X_val)/len(features):.1f}%)")
print(f"   Test      : {X_test.shape[0]:>6d} samples ({100*len(X_test)/len(features):.1f}%)")
print(f"\n   Train class dist: Benign={np.sum(y_train==0)}, Malignant={np.sum(y_train==1)}")
print(f"   Val class dist  : Benign={np.sum(y_val==0)}, Malignant={np.sum(y_val==1)}")
print(f"   Test class dist : Benign={np.sum(y_test==0)}, Malignant={np.sum(y_test==1)}")

## 4. Reshape Data for 1D CNN

The 1D CNN expects input of shape `(batch_size, sequence_length, channels)`. We reshape our feature vectors accordingly:
- **Option A:** Treat as single-channel 1D sequence: `(N, 90, 1)`
- **Option B:** Group features into multi-channel format: `(N, 10, 9)` - 10 time steps, 9 feature channels

We use **Option B** as it preserves the temporal structure of the DE features.

In [ ]:
n_features = features.shape[1]

# Strategy: Reshape to (N, timesteps, channels)
# We have 90 features = 10 time points x 9 feature groups
# Feature groups: ODE coeffs (padded) + Euler ODE + RK4 ODE + Euler res + RK4 res + 
#                 Euler IDE + RK4 IDE + Euler IDE res + RK4 IDE res

TIMESTEPS = 10
CHANNELS = n_features // TIMESTEPS

if n_features % TIMESTEPS != 0:
    # Pad features to make divisible
    pad_size = TIMESTEPS - (n_features % TIMESTEPS)
    CHANNELS = (n_features + pad_size) // TIMESTEPS
    X_train_pad = np.pad(X_train, ((0, 0), (0, pad_size)), mode='constant')
    X_val_pad = np.pad(X_val, ((0, 0), (0, pad_size)), mode='constant')
    X_test_pad = np.pad(X_test, ((0, 0), (0, pad_size)), mode='constant')
else:
    X_train_pad = X_train
    X_val_pad = X_val
    X_test_pad = X_test

# Reshape to (N, timesteps, channels)
X_train_cnn = X_train_pad.reshape(-1, TIMESTEPS, CHANNELS)
X_val_cnn = X_val_pad.reshape(-1, TIMESTEPS, CHANNELS)
X_test_cnn = X_test_pad.reshape(-1, TIMESTEPS, CHANNELS)

print(f"CNN Input Shape: ({TIMESTEPS}, {CHANNELS})")
print(f"   X_train: {X_train_cnn.shape}")
print(f"   X_val  : {X_val_cnn.shape}")
print(f"   X_test : {X_test_cnn.shape}")

## 5. Compute Class Weights

Handle class imbalance by computing inverse-frequency class weights.

In [ ]:
# Compute class weights to handle imbalance
class_weights_array = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: w for i, w in enumerate(class_weights_array)}

print(f"⚖️ Class Weights:")
print(f"   Benign (0)   : {class_weight_dict[0]:.4f}")
print(f"   Malignant (1): {class_weight_dict[1]:.4f}")

## 6. Build 1D CNN Architecture

### Architecture Overview

```
Input (10, 9)
   │
   ├── Conv1D(64, kernel=3) + BatchNorm + ReLU
   ├── Conv1D(128, kernel=3) + BatchNorm + ReLU + Dropout(0.3)
   ├── Conv1D(256, kernel=3) + BatchNorm + ReLU + Dropout(0.3)
   ├── GlobalAveragePooling1D
   ├── Dense(128) + ReLU + Dropout(0.5)
   ├── Dense(64) + ReLU + Dropout(0.3)
   └── Dense(1) + Sigmoid
```

In [ ]:
def build_cnn_model(input_shape, learning_rate=1e-3):
    """
    Build a 1D CNN model for binary classification.
    
    Parameters:
        input_shape: tuple (timesteps, channels)
        learning_rate: optimizer learning rate
    
    Returns:
        Compiled Keras model
    """
    model = models.Sequential([
        # Input layer
        layers.Input(shape=input_shape),
        
        # Conv Block 1
        layers.Conv1D(64, kernel_size=3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        
        # Conv Block 2
        layers.Conv1D(128, kernel_size=3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        
        # Conv Block 3
        layers.Conv1D(256, kernel_size=3, padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        
        # Global pooling
        layers.GlobalAveragePooling1D(),
        
        # Dense classification head
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        
        # Output
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Compile
    model.compile(
        optimizer=optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='auc')
        ]
    )
    
    return model


# Build the model
input_shape = (TIMESTEPS, CHANNELS)
model = build_cnn_model(input_shape, learning_rate=1e-3)

# Model summary
print("\n" + "="*60)
print(" 1D CNN MODEL ARCHITECTURE")
print("="*60)
model.summary()

In [ ]:
# Visualize model architecture (save as image)
try:
    arch_path = os.path.join(MODEL_DIR, 'cnn_architecture.png')
    plot_model(model, to_file=arch_path, show_shapes=True, show_layer_names=True,
               dpi=100, expand_nested=True)
    print(f"✅ Model architecture saved to: {arch_path}")
except Exception as e:
    print(f"⚠️ Could not save model architecture image: {e}")
    print("   Install graphviz and pydot: pip install pydot graphviz")

## 7. Define Training Callbacks

Set up callbacks for:
- **EarlyStopping** - Stop training when validation loss stops improving
- **ReduceLROnPlateau** - Reduce learning rate when validation loss plateaus
- **ModelCheckpoint** - Save the best model based on validation AUC

In [ ]:
# Training hyperparameters
EPOCHS = 100
BATCH_SIZE = 32

# Callbacks
best_model_path = os.path.join(MODEL_DIR, 'best_cnn_model.keras')

callback_list = [
    callbacks.EarlyStopping(
        monitor='val_auc',
        patience=15,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=best_model_path,
        monitor='val_auc',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

print(f"⚙️ Training Configuration:")
print(f"   Epochs     : {EPOCHS}")
print(f"   Batch Size : {BATCH_SIZE}")
print(f"   Optimizer  : Adam (lr=1e-3)")
print(f"   Loss       : Binary Crossentropy")
print(f"   Callbacks  : EarlyStopping, ReduceLROnPlateau, ModelCheckpoint")

## 8. Train the Model

In [ ]:
print("\n" + "="*60)
print(" TRAINING CNN CLASSIFIER")
print("="*60 + "\n")

history = model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callback_list,
    verbose=1
)

print("\n✅ Training completed!")
print(f"   Total epochs trained: {len(history.history['loss'])}")
print(f"   Best val AUC: {max(history.history['val_auc']):.4f}")
print(f"   Best val accuracy: {max(history.history['val_accuracy']):.4f}")

## 9. Training History Visualization

In [ ]:
def plot_training_history(history):
    """
    Plot comprehensive training curves.
    """
    metrics = ['loss', 'accuracy', 'auc', 'precision', 'recall']
    titles = ['Binary Crossentropy Loss', 'Accuracy', 'AUC-ROC', 'Precision', 'Recall']
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        if idx >= len(axes):
            break
        
        train_vals = history.history.get(metric, [])
        val_vals = history.history.get(f'val_{metric}', [])
        epochs = range(1, len(train_vals) + 1)
        
        if train_vals:
            axes[idx].plot(epochs, train_vals, 'b-', linewidth=2, label='Training', alpha=0.8)
        if val_vals:
            axes[idx].plot(epochs, val_vals, 'r-', linewidth=2, label='Validation', alpha=0.8)
        
        axes[idx].set_title(title, fontsize=13, fontweight='bold')
        axes[idx].set_xlabel('Epoch')
        axes[idx].set_ylabel(title)
        axes[idx].legend(fontsize=10)
        axes[idx].grid(True, alpha=0.3)
    
    # Learning rate plot
    if 'lr' in history.history:
        lr_vals = history.history['lr']
        epochs_lr = range(1, len(lr_vals) + 1)
        axes[5].plot(epochs_lr, lr_vals, 'g-', linewidth=2)
        axes[5].set_title('Learning Rate', fontsize=13, fontweight='bold')
        axes[5].set_xlabel('Epoch')
        axes[5].set_ylabel('LR')
        axes[5].set_yscale('log')
        axes[5].grid(True, alpha=0.3)
    else:
        axes[5].axis('off')
    
    fig.suptitle('CNN Training History', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
    plt.show()


plot_training_history(history)

## 10. Quick Evaluation on Test Set

In [ ]:
# Load best model
if os.path.exists(best_model_path):
    best_model = keras.models.load_model(best_model_path)
    print(f"✅ Loaded best model from: {best_model_path}")
else:
    best_model = model
    print("ℹ️ Using current model (no checkpoint saved)")

# Evaluate on test set
print("\n" + "="*60)
print(" TEST SET EVALUATION")
print("="*60)

test_results = best_model.evaluate(X_test_cnn, y_test, verbose=0)
metric_names = best_model.metrics_names

print(f"\n📊 Test Set Results:")
for name, value in zip(metric_names, test_results):
    print(f"   {name:>12s}: {value:.4f}")

# Generate predictions
y_pred_proba = best_model.predict(X_test_cnn, verbose=0).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

print(f"\n   Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign (LGG)', 'Malignant (HGG)']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
plt.colorbar(im, ax=ax, shrink=0.8)

classes = ['Benign\n(LGG)', 'Malignant\n(HGG)']
tick_marks = [0, 1]
ax.set_xticks(tick_marks)
ax.set_xticklabels(classes, fontsize=12)
ax.set_yticks(tick_marks)
ax.set_yticklabels(classes, fontsize=12)

# Annotate cells
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, f'{cm[i, j]}\n({100*cm[i,j]/cm.sum():.1f}%)',
                ha='center', va='center', fontsize=14, fontweight='bold', color=color)

ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
ax.set_ylabel('True Label', fontsize=13, fontweight='bold')
ax.set_title('Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Save Model and Artifacts

In [ ]:
# Save the final model
final_model_path = os.path.join(MODEL_DIR, 'final_cnn_model.keras')
best_model.save(final_model_path)

# Save test predictions for Notebook 05
predictions_path = os.path.join(OUTPUT_DIR, 'test_predictions.npz')
np.savez_compressed(
    predictions_path,
    y_test=y_test,
    y_pred=y_pred,
    y_pred_proba=y_pred_proba,
    X_test=X_test,
    X_test_cnn=X_test_cnn
)

# Save training history
history_df = pd.DataFrame(history.history)
history_df.to_csv(os.path.join(OUTPUT_DIR, 'training_history.csv'), index=False)

# Save model config
model_config = {
    'input_shape': list(input_shape),
    'timesteps': TIMESTEPS,
    'channels': CHANNELS,
    'epochs_trained': len(history.history['loss']),
    'best_val_auc': float(max(history.history['val_auc'])),
    'best_val_accuracy': float(max(history.history['val_accuracy'])),
    'test_accuracy': float(test_results[metric_names.index('accuracy')]),
    'test_auc': float(test_results[metric_names.index('auc')]),
    'total_params': int(model.count_params()),
    'model_path': final_model_path,
    'batch_size': BATCH_SIZE
}

with open(os.path.join(MODEL_DIR, 'model_config.json'), 'w') as f:
    json.dump(model_config, f, indent=2)

print("✅ All artifacts saved!")
print(f"   Final model   : {final_model_path}")
print(f"   Best model    : {best_model_path}")
print(f"   Predictions   : {predictions_path}")
print(f"   History       : {os.path.join(OUTPUT_DIR, 'training_history.csv')}")
print(f"   Model config  : {os.path.join(MODEL_DIR, 'model_config.json')}")

---

## ✅ Summary

In this notebook, we built and trained a 1D CNN classifier:

### Model Details
| Aspect | Value |
|--------|-------|
| Architecture | 3×Conv1D + GlobalAvgPool + 2×Dense |
| Input Shape | (10, 9) - 10 timesteps, 9 channels |
| Total Parameters | ~see summary above |
| Loss Function | Binary Crossentropy |
| Optimizer | Adam with ReduceLROnPlateau |
| Regularization | BatchNorm + Dropout (0.3-0.5) |

### Training Results
- Best validation AUC achieved during training
- Class weighting applied to handle HGG/LGG imbalance
- EarlyStopping prevented overfitting

### ➡️ Next: [Notebook 05 - Evaluation & Results](./05_Evaluation_and_Results.ipynb)

Comprehensive evaluation with ROC curves, precision-recall analysis, and method comparison.